# 3. Anonymization

Six rules, in order: outlet signature in the titles → `[OUTLET]`; e-mails → `[EMAIL]`; links → `[URL]`;
@accounts → `[ACCOUNT]`; the outlet's own authors → `[AUTHOR]`; the outlet name from its domain → `[OUTLET]`
(see `anonymization.py`). The checks are automatic.

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import re

import anonymization as AN

In [ ]:
art = pd.read_csv(C.OUT / "articles.csv.gz")
anon, counts, found = AN.anonymize(art)
RULES = ["signature", "email", "url", "account", "author", "name"]
print(f"articles with at least one replacement: {((counts[RULES].sum(axis=1) + counts[[f'title_{r}' for r in RULES]].sum(axis=1)) > 0).mean():.1%}")
pd.DataFrame({"articles touched (%)": [(counts[r] > 0).mean() * 100 for r in RULES],
              "replacements": [counts[r].sum() for r in RULES],
              "titles touched (%)": [(counts[f"title_{r}"] > 0).mean() * 100 for r in RULES]}, index=RULES).round(1)

In [ ]:
print("common-word names:", ", ".join(sorted(found["common"])))
print("written in lowercase by their own outlet:", ", ".join(sorted(found["lowercase"])))
print(f"signatures: {len(found['signatures'])} outlets; own authors: {sum(map(len, found['authors'].values())):,}")
for token in ["[OUTLET]", "[AUTHOR]"]:
    for e in AN.examples(anon.text, re.compile(re.escape(token)), n=3):
        print(" ·", e)

## Check: nothing left

In [ ]:
left = AN.leftovers(anon, found)
distinct = left[~left.common_word]
print(f"distinctive names ({len(distinct)} outlets): articles with the name as a word {distinct.left_as_word.sum()}, "
      f"inside other words {distinct.left_inside_words.sum()}")
assert distinct.left_as_word.sum() == 0
left[left.common_word | (left.left_inside_words > 0)]

In [ ]:
anon[["article_id", "domain", "title", "date", "text", "score", "label", "level"]].to_csv(C.OUT / "articles_anonymous.csv.gz", index=False)
left.to_csv(C.OUT / "anonymization_check.csv", index=False)
print("saved", C.OUT / "articles_anonymous.csv.gz")